# Week 7 — Phase-aware GNN 与高召回筛查策略

本周把 Week 5/6 的 **flattened feature-vector model** 升级为 **topology-aware graph model**。

我们仍然预测同一个监督学习目标：

\[
(x_t,c) \rightarrow y_{t,c},\quad s_{t,c}
\]

但输入表示从一行 tabular features 变成带有三相通道和 contingency mask 的图：

\[
G_{t,c}=\bigl(\mathcal N,\mathcal E, X_t, E_c, g_{t,c}\bigr)
\]

其中 $X_t$ 是 bus-level phase-channel node features，$E_c$ 是 edge features + outage mask，$g_{t,c}$ 是 scenario/global/contingency features。

> 教学重点：本周不把 GNN 当作“必然更准”的模型，而是把它当作一种更符合电网拓扑结构的建模方式。小数据下，MLP 可能表现更好；这恰好可以训练学生正确解读实验结果、做 OOD 检查和写 limitation。


## 本周学习目标

完成本 Notebook 后，学生应能做到：

1. 把三相微电网表示为 bus-level graph；
2. 解释 phase-channel node feature 与 phase-level node graph 的区别；
3. 用 outage mask 编码 line outage，用 node/global flags 编码 PV/BESS/PCC trip；
4. 实现一个不依赖 PyTorch Geometric 的 message-passing GNN；
5. 证明 graph tensor、mask、split、loss 和 prediction 过程没有明显代码错误；
6. 使用 high-recall threshold 和 top-k ranking 评价 AI-assisted N-1 screening；
7. 用 random holdout、scenario-group CV、contingency-group CV 比较泛化能力。


## 1. 本周 Notebook 的 proof / cross-validation 目标

本 Notebook 不只训练模型，还要验证代码是否可信：

1. **Graph topology proof**：base feeder connected and radial。
2. **Graph tensor proof**：node / edge / global tensors finite and dimensionally correct。
3. **Contingency mask proof**：line outage exactly removes one line; PV/BESS/PCC trip marks the correct bus。
4. **No-leakage proof**：post-contingency results and labels are not used as model inputs。
5. **Forward / gradient proof**：model output shapes, loss, and gradients are finite。
6. **Permutation-invariance proof**：node ordering changes should not change graph-level prediction。
7. **Split proof**：random split, scenario-group split, contingency-group split have no overlap。
8. **Screening proof**：threshold is chosen only on validation data; top-k ranking uses out-of-fold scores。

In [ ]:
# ============================================================
# 2. Imports and configuration
# ============================================================
from __future__ import annotations

import json
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

SEED = 202607
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(False)

ROOT = Path("/mnt/data")
INPUT_WEEK5 = ROOT / "week5_outputs_complete" / "week5_ml_input_dataset.csv"
INPUT_CONT = ROOT / "week4_outputs_complete" / "week4_contingency_table.csv"
OUTDIR = ROOT / "week7_outputs_complete"
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET_RECALL = 0.95
DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Output directory:", OUTDIR)
assert INPUT_WEEK5.exists(), f"Missing {INPUT_WEEK5}. Run Week 5 first."
assert INPUT_CONT.exists(), f"Missing {INPUT_CONT}. Run Week 4 first."

In [ ]:
# ============================================================
# 3. Load Week 5 / Week 4 datasets
# ============================================================
df = pd.read_csv(INPUT_WEEK5)
contingency_table = pd.read_csv(INPUT_CONT)

# The Week 5 dataset already contains a row_id and merged base-case features.
assert "row_id" in df.columns
assert "violation_label" in df.columns
assert "severity_score" in df.columns
assert df["row_id"].is_unique
assert set(df["violation_label"].unique()).issubset({0, 1, False, True})

df["violation_label"] = df["violation_label"].astype(int)
df["severity_score"] = df["severity_score"].astype(float)

print("Dataset shape:", df.shape)
print("Scenarios:", df["scenario_id"].nunique())
print("Contingencies:", df["contingency_id"].nunique())
print("Unsafe samples:", int(df["violation_label"].sum()))
print("Safe samples:", int((1-df["violation_label"]).sum()))

display(df[["row_id", "scenario_id", "contingency_id", "contingency_type", "violation_label", "severity_score"]].head())
display(contingency_table)

## 4. Define the teaching microgrid graph

我们复用 Week 4 的教学 feeder。这个图有 9 个 buses 和 8 条 lines。Week 7 不重新运行 pandapower，而是使用 Week 4/5 的数据结果训练筛查模型。

GNN 的输入图结构如下：

```text
PCC_0p4kV -- Main_bus -- Critical_load_bus -- EV_bus -- Remote_bus
                   |              |
                   |              Medical_bus
                   |
              Residential_bus -- PV_A_bus
                   |
                   PV_C_bus
```

注意：line outage contingency 只选了 Week 4 中的部分线路，但 GNN 的 base graph 包含完整 feeder 拓扑。

In [ ]:
# ============================================================
# 4. Static graph metadata
# ============================================================
PHASES = ["a", "b", "c"]

BUS_NAMES = [
    "PCC_0p4kV",
    "Main_bus",
    "Critical_load_bus",
    "Residential_bus",
    "PV_A_bus",
    "PV_C_bus",
    "EV_bus",
    "Remote_bus",
    "Medical_bus",
]
BUS_INDEX = {name: i for i, name in enumerate(BUS_NAMES)}

LINE_SPECS = [
    (0, "PCC_0p4kV", "Main_bus", 0.020, 0.25, "L0_PCC_to_main"),
    (1, "Main_bus", "Critical_load_bus", 0.030, 0.25, "L1_main_to_critical"),
    (2, "Main_bus", "Residential_bus", 0.035, 0.25, "L2_main_to_residential"),
    (3, "Residential_bus", "PV_A_bus", 0.030, 0.25, "L3_residential_to_pvA"),
    (4, "Residential_bus", "PV_C_bus", 0.030, 0.25, "L4_residential_to_pvC"),
    (5, "Critical_load_bus", "EV_bus", 0.035, 0.25, "L5_critical_to_ev"),
    (6, "EV_bus", "Remote_bus", 0.040, 0.25, "L6_ev_to_remote"),
    (7, "Critical_load_bus", "Medical_bus", 0.020, 0.25, "L7_critical_to_medical"),
]
LINE_BY_ID = {idx: (idx, f, t, length, imax, name) for idx, f, t, length, imax, name in LINE_SPECS}

BASE_LOADS = {
    "Critical_load_bus": {"p": (0.006, 0.005, 0.006), "q": (0.0012, 0.0010, 0.0012), "critical": True},
    "Residential_bus": {"p": (0.005, 0.007, 0.004), "q": (0.0010, 0.0014, 0.0008), "critical": False},
    "EV_bus": {"p": (0.003, 0.008, 0.003), "q": (0.0006, 0.0016, 0.0006), "critical": False},
    "Remote_bus": {"p": (0.004, 0.002, 0.005), "q": (0.0008, 0.0004, 0.0010), "critical": False},
    "Medical_bus": {"p": (0.003, 0.004, 0.003), "q": (0.0006, 0.0008, 0.0006), "critical": True},
}

PV_SPECS = [
    {"name": "PV_single_phase_A", "bus": "PV_A_bus", "phase": "a", "rating_mw": 0.006, "element_index": 0},
    {"name": "PV_single_phase_C", "bus": "PV_C_bus", "phase": "c", "rating_mw": 0.006, "element_index": 1},
    {"name": "PV_three_phase_remote", "bus": "Remote_bus", "phase": "abc", "rating_mw": 0.006, "element_index": 2},
]
BESS_BUS = "Critical_load_bus"
PCC_BUS = "PCC_0p4kV"

# Undirected base graph for proofs and plotting.
G_BASE = nx.Graph()
G_BASE.add_nodes_from(range(len(BUS_NAMES)))
for line_id, f, t, length, imax, name in LINE_SPECS:
    G_BASE.add_edge(BUS_INDEX[f], BUS_INDEX[t], line_id=line_id, name=name, length_km=length, max_i_ka=imax)

print("Number of buses:", G_BASE.number_of_nodes())
print("Number of lines:", G_BASE.number_of_edges())
print("Connected:", nx.is_connected(G_BASE))
print("Radial:", G_BASE.number_of_edges() == G_BASE.number_of_nodes() - 1)

assert nx.is_connected(G_BASE), "Teaching feeder graph should be connected."
assert G_BASE.number_of_edges() == G_BASE.number_of_nodes() - 1, "Teaching feeder graph should be radial."

display(pd.DataFrame(LINE_SPECS, columns=["line_id", "from_bus", "to_bus", "length_km", "max_i_ka", "name"]))

## 5. Convert each \((x_t,c)\) row into a graph sample

每个样本包含：

```text
X_node:  [n_bus, n_node_features]
E_edge:  [2*n_line, n_edge_features]
global:  [n_global_features]
y:       binary violation label
severity: severity score
```

为什么 `2*n_line`？因为我们把每条无向线路表示为两个有向 message-passing directions。

In [ ]:
# ============================================================
# 5. Feature construction
# ============================================================
NODE_FEATURES = [
    "is_pcc", "is_main", "is_critical", "is_load_bus", "is_pv_bus", "is_bess_bus",
    "depth_from_pcc",
    "p_load_a", "p_load_b", "p_load_c",
    "p_pv_a", "p_pv_b", "p_pv_c",
    "p_bess_dis_a", "p_bess_dis_b", "p_bess_dis_c",
    "p_bess_ch_a", "p_bess_ch_b", "p_bess_ch_c",
    "cont_line_endpoint", "cont_pv_trip", "cont_bess_trip", "cont_pcc_outage",
    "base_min_vm_pu", "base_max_loading_frac", "base_max_vuf_frac",
]
EDGE_FEATURES = [
    "availability", "outage_flag", "length_km", "max_i_ka", "from_depth", "to_depth",
]
GLOBAL_FEATURES = [
    "load_scale", "phase_a_factor", "phase_b_factor", "phase_c_factor", "pv_scale", "bess_p_discharge_mw",
    "base_min_vm_pu", "base_max_vm_pu", "base_max_line_loading_frac", "base_max_vuf_frac", "base_p_grid_mw",
    "ctype_line_outage", "ctype_pv_trip", "ctype_bess_trip", "ctype_pcc_outage",
    "contingency_element_norm",
]

# Base edge index, directed.
EDGE_INDEX_LIST = []
BASE_EDGE_META = []
for line_id, f, t, length, imax, name in LINE_SPECS:
    u, v = BUS_INDEX[f], BUS_INDEX[t]
    EDGE_INDEX_LIST.append((u, v, line_id, length, imax))
    EDGE_INDEX_LIST.append((v, u, line_id, length, imax))

EDGE_INDEX = torch.tensor([[u for u, v, *_ in EDGE_INDEX_LIST], [v for u, v, *_ in EDGE_INDEX_LIST]], dtype=torch.long)
DEPTHS = dict(nx.shortest_path_length(G_BASE, source=BUS_INDEX[PCC_BUS]))
DEPTH_BY_BUS = {bus: float(depth) for bus, depth in DEPTHS.items()}
MAX_DEPTH = max(DEPTH_BY_BUS.values())


def ctype_one_hot(ctype: str) -> Dict[str, float]:
    return {
        "ctype_line_outage": 1.0 if ctype == "line_outage" else 0.0,
        "ctype_pv_trip": 1.0 if ctype == "pv_trip" else 0.0,
        "ctype_bess_trip": 1.0 if ctype == "bess_trip" else 0.0,
        "ctype_pcc_outage": 1.0 if ctype == "pcc_outage" else 0.0,
    }


def scenario_phase_loads(row: pd.Series) -> Dict[str, np.ndarray]:
    """Return per-bus active load MW vector [a,b,c] under the scenario."""
    load_scale = float(row["load_scale"])
    phase_factors = np.array([float(row[f"phase_{ph}_factor"]) for ph in PHASES])
    loads = {name: np.zeros(3, dtype=float) for name in BUS_NAMES}
    for bus_name, spec in BASE_LOADS.items():
        loads[bus_name] += np.array(spec["p"], dtype=float) * load_scale * phase_factors
    # Static BESS setpoint convention used in Week 3/4: p>0 discharges; p<0 charges.
    p_bess = float(row["bess_p_discharge_mw"])
    if p_bess < 0:
        loads[BESS_BUS] += np.ones(3) * (-p_bess / 3.0)
    return loads


def scenario_phase_pv(row: pd.Series) -> Dict[str, np.ndarray]:
    """Return per-bus active PV/BESS discharge injections MW vector [a,b,c]."""
    pv_scale = float(row["pv_scale"])
    injections = {name: np.zeros(3, dtype=float) for name in BUS_NAMES}
    for pv in PV_SPECS:
        bus = pv["bus"]
        phase = pv["phase"]
        rating = float(pv["rating_mw"])
        if phase == "abc":
            injections[bus] += np.ones(3) * rating * pv_scale / 3.0
        else:
            injections[bus][PHASES.index(phase)] += rating * pv_scale
    p_bess = float(row["bess_p_discharge_mw"])
    if p_bess > 0:
        injections[BESS_BUS] += np.ones(3) * (p_bess / 3.0)
    return injections


def build_graph_sample(row: pd.Series) -> Dict[str, np.ndarray]:
    ctype = str(row["contingency_type"])
    element_index_raw = row["element_index"]
    try:
        element_index = int(element_index_raw)
    except Exception:
        element_index = -1

    loads = scenario_phase_loads(row)
    pvs = scenario_phase_pv(row)

    X = np.zeros((len(BUS_NAMES), len(NODE_FEATURES)), dtype=np.float32)
    for bus_name, bus_idx in BUS_INDEX.items():
        # Role flags.
        X[bus_idx, NODE_FEATURES.index("is_pcc")] = 1.0 if bus_name == PCC_BUS else 0.0
        X[bus_idx, NODE_FEATURES.index("is_main")] = 1.0 if bus_name == "Main_bus" else 0.0
        X[bus_idx, NODE_FEATURES.index("is_critical")] = 1.0 if BASE_LOADS.get(bus_name, {}).get("critical", False) else 0.0
        X[bus_idx, NODE_FEATURES.index("is_load_bus")] = 1.0 if bus_name in BASE_LOADS else 0.0
        X[bus_idx, NODE_FEATURES.index("is_pv_bus")] = 1.0 if any(pv["bus"] == bus_name for pv in PV_SPECS) else 0.0
        X[bus_idx, NODE_FEATURES.index("is_bess_bus")] = 1.0 if bus_name == BESS_BUS else 0.0
        X[bus_idx, NODE_FEATURES.index("depth_from_pcc")] = DEPTH_BY_BUS[bus_idx] / MAX_DEPTH

        # Phase-channel active power features.
        for k, ph in enumerate(PHASES):
            X[bus_idx, NODE_FEATURES.index(f"p_load_{ph}")] = loads[bus_name][k]
            X[bus_idx, NODE_FEATURES.index(f"p_pv_{ph}")] = pvs[bus_name][k]
        p_bess = float(row["bess_p_discharge_mw"])
        for k, ph in enumerate(PHASES):
            if bus_name == BESS_BUS and p_bess > 0:
                X[bus_idx, NODE_FEATURES.index(f"p_bess_dis_{ph}")] = p_bess / 3.0
            if bus_name == BESS_BUS and p_bess < 0:
                X[bus_idx, NODE_FEATURES.index(f"p_bess_ch_{ph}")] = -p_bess / 3.0

        # Broadcast base-case context to each node. This is allowed because it is known before N-1 verification.
        X[bus_idx, NODE_FEATURES.index("base_min_vm_pu")] = float(row["base_min_vm_pu"])
        X[bus_idx, NODE_FEATURES.index("base_max_loading_frac")] = float(row["base_max_line_loading_percent"]) / 100.0
        X[bus_idx, NODE_FEATURES.index("base_max_vuf_frac")] = float(row["base_max_vuf_percent"]) / 100.0

    # Contingency node flags.
    if ctype == "line_outage" and element_index in LINE_BY_ID:
        _, f, t, *_ = LINE_BY_ID[element_index]
        X[BUS_INDEX[f], NODE_FEATURES.index("cont_line_endpoint")] = 1.0
        X[BUS_INDEX[t], NODE_FEATURES.index("cont_line_endpoint")] = 1.0
    elif ctype == "pv_trip":
        matches = [pv for pv in PV_SPECS if int(pv["element_index"]) == element_index]
        if matches:
            X[BUS_INDEX[matches[0]["bus"]], NODE_FEATURES.index("cont_pv_trip")] = 1.0
    elif ctype == "bess_trip":
        X[BUS_INDEX[BESS_BUS], NODE_FEATURES.index("cont_bess_trip")] = 1.0
    elif ctype == "pcc_outage":
        X[BUS_INDEX[PCC_BUS], NODE_FEATURES.index("cont_pcc_outage")] = 1.0

    # Edge features. Keep availability and outage_flag unscaled later.
    E = np.zeros((len(EDGE_INDEX_LIST), len(EDGE_FEATURES)), dtype=np.float32)
    for d, (u, v, line_id, length, imax) in enumerate(EDGE_INDEX_LIST):
        is_outaged = ctype == "line_outage" and element_index == int(line_id)
        E[d, EDGE_FEATURES.index("availability")] = 0.0 if is_outaged else 1.0
        E[d, EDGE_FEATURES.index("outage_flag")] = 1.0 if is_outaged else 0.0
        E[d, EDGE_FEATURES.index("length_km")] = length
        E[d, EDGE_FEATURES.index("max_i_ka")] = imax
        E[d, EDGE_FEATURES.index("from_depth")] = DEPTH_BY_BUS[u] / MAX_DEPTH
        E[d, EDGE_FEATURES.index("to_depth")] = DEPTH_BY_BUS[v] / MAX_DEPTH

    oh = ctype_one_hot(ctype)
    element_norm = max(float(element_index), 0.0) / 10.0
    g = np.array([
        float(row["load_scale"]), float(row["phase_a_factor"]), float(row["phase_b_factor"]), float(row["phase_c_factor"]),
        float(row["pv_scale"]), float(row["bess_p_discharge_mw"]),
        float(row["base_min_vm_pu"]), float(row["base_max_vm_pu"]),
        float(row["base_max_line_loading_percent"]) / 100.0, float(row["base_max_vuf_percent"]) / 100.0,
        float(row["base_p_grid_mw"]),
        oh["ctype_line_outage"], oh["ctype_pv_trip"], oh["ctype_bess_trip"], oh["ctype_pcc_outage"],
        element_norm,
    ], dtype=np.float32)

    return {"x": X, "edge_attr": E, "global": g}

samples = [build_graph_sample(row) for _, row in df.iterrows()]
X_all = np.stack([s["x"] for s in samples])
E_all = np.stack([s["edge_attr"] for s in samples])
G_all = np.stack([s["global"] for s in samples])
y_all = df["violation_label"].to_numpy(dtype=np.float32)
sev_all = df["severity_score"].to_numpy(dtype=np.float32)

print("X_all:", X_all.shape)
print("E_all:", E_all.shape)
print("G_all:", G_all.shape)
print("y_all:", y_all.shape)
assert X_all.shape == (len(df), len(BUS_NAMES), len(NODE_FEATURES))
assert E_all.shape == (len(df), 2 * len(LINE_SPECS), len(EDGE_FEATURES))
assert G_all.shape == (len(df), len(GLOBAL_FEATURES))
assert np.isfinite(X_all).all()
assert np.isfinite(E_all).all()
assert np.isfinite(G_all).all()

In [ ]:
# ============================================================
# 6. Graph tensor and no-leakage proofs
# ============================================================
validation_records = []

def add_check(name: str, passed: bool, detail: str = ""):
    validation_records.append({"check": name, "passed": bool(passed), "detail": detail})
    assert passed, f"Validation failed: {name}. {detail}"

# Topology proof.
add_check("base_graph_connected", nx.is_connected(G_BASE), "Base graph is connected.")
add_check("base_graph_radial", G_BASE.number_of_edges() == G_BASE.number_of_nodes() - 1, "Edges = buses - 1.")
add_check("directed_edge_count", EDGE_INDEX.shape[1] == 2 * len(LINE_SPECS), "Two directed edges per line.")

# Graph tensor proof.
add_check("node_tensor_finite", np.isfinite(X_all).all(), f"shape={X_all.shape}")
add_check("edge_tensor_finite", np.isfinite(E_all).all(), f"shape={E_all.shape}")
add_check("global_tensor_finite", np.isfinite(G_all).all(), f"shape={G_all.shape}")

# Contingency mask proof.
line_rows = df["contingency_type"].eq("line_outage").to_numpy()
line_outage_counts = E_all[line_rows, :, EDGE_FEATURES.index("outage_flag")].sum(axis=1)
add_check("line_outage_mask_exactly_two_directed_edges", np.all(line_outage_counts == 2), "One undirected line = two directed edges.")
non_line_rows = ~line_rows
non_line_outage_counts = E_all[non_line_rows, :, EDGE_FEATURES.index("outage_flag")].sum(axis=1)
add_check("non_line_contingencies_do_not_remove_edges", np.all(non_line_outage_counts == 0), "PV/BESS/PCC do not change line availability.")

pv_rows = df["contingency_type"].eq("pv_trip").to_numpy()
pv_flags = X_all[pv_rows, :, NODE_FEATURES.index("cont_pv_trip")].sum(axis=1)
add_check("pv_trip_marks_one_bus", np.all(pv_flags == 1), "Each PV trip marks exactly one PV bus.")

bess_rows = df["contingency_type"].eq("bess_trip").to_numpy()
bess_flags = X_all[bess_rows, :, NODE_FEATURES.index("cont_bess_trip")].sum(axis=1)
add_check("bess_trip_marks_one_bus", np.all(bess_flags == 1), "BESS trip marks the BESS bus.")

pcc_rows = df["contingency_type"].eq("pcc_outage").to_numpy()
pcc_flags = X_all[pcc_rows, :, NODE_FEATURES.index("cont_pcc_outage")].sum(axis=1)
add_check("pcc_outage_marks_one_bus", np.all(pcc_flags == 1), "PCC outage marks the PCC bus.")

# No-leakage proof: these columns are forbidden as model inputs.
FORBIDDEN_INPUT_COLUMNS = {
    "converged", "min_vm_pu", "max_vm_pu", "max_vuf_percent", "max_line_loading_percent",
    "unserved_load_mw", "critical_unserved_load_mw",
    "voltage_low_violation", "voltage_high_violation", "loading_violation", "vuf_violation",
    "non_convergence_violation", "service_loss_violation", "critical_service_loss_violation",
    "violation_label", "severity_score",
}
USED_RAW_COLUMNS = {
    "load_scale", "phase_a_factor", "phase_b_factor", "phase_c_factor", "pv_scale", "bess_p_discharge_mw",
    "base_min_vm_pu", "base_max_vm_pu", "base_max_line_loading_percent", "base_max_vuf_percent", "base_p_grid_mw",
    "contingency_type", "element_index",
}
add_check("no_post_contingency_feature_columns", len(FORBIDDEN_INPUT_COLUMNS & USED_RAW_COLUMNS) == 0, str(FORBIDDEN_INPUT_COLUMNS & USED_RAW_COLUMNS))

# Save graph sample metadata for inspection.
graph_sample_table = df[["row_id", "scenario_id", "contingency_id", "contingency_type", "violation_label", "severity_score"]].copy()
graph_sample_table["n_outaged_directed_edges"] = E_all[:, :, EDGE_FEATURES.index("outage_flag")].sum(axis=1).astype(int)
graph_sample_table["n_marked_contingency_nodes"] = (
    X_all[:, :, [NODE_FEATURES.index("cont_line_endpoint"), NODE_FEATURES.index("cont_pv_trip"), NODE_FEATURES.index("cont_bess_trip"), NODE_FEATURES.index("cont_pcc_outage")]].sum(axis=(1, 2))
).astype(int)
graph_sample_table.to_csv(OUTDIR / "week7_graph_sample_table.csv", index=False)

display(graph_sample_table.head(12))
print("Validation checks so far:", len(validation_records))

## 7. Train / validation / test split and preprocessing

这里使用随机 holdout split 作为第一组结果，然后再做 scenario-group CV 和 contingency-group CV。

为了防止 preprocessing leakage：

- node scaler 只在 training samples 的所有 nodes 上 fit；
- global scaler 只在 training samples 上 fit；
- edge continuous-feature scaler 只在 training samples 的 edges 上 fit；
- edge availability / outage flag 保持 0/1，不做标准化，因为 message passing 中要直接用 availability mask。

In [ ]:
# ============================================================
# 7. Split and preprocessing
# ============================================================
idx_all = np.arange(len(df))
idx_trainval, idx_test = train_test_split(idx_all, test_size=0.25, random_state=SEED, stratify=y_all)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.30, random_state=SEED, stratify=y_all[idx_trainval])

add_check("split_disjoint_train_val", len(set(idx_train) & set(idx_val)) == 0)
add_check("split_disjoint_train_test", len(set(idx_train) & set(idx_test)) == 0)
add_check("split_disjoint_val_test", len(set(idx_val) & set(idx_test)) == 0)

print("Train / Val / Test sizes:", len(idx_train), len(idx_val), len(idx_test))
print("Unsafe in train/val/test:", int(y_all[idx_train].sum()), int(y_all[idx_val].sum()), int(y_all[idx_test].sum()))

@dataclass
class GraphScalers:
    node_scaler: StandardScaler
    edge_scaler: StandardScaler
    global_scaler: StandardScaler


def fit_graph_scalers(train_idx: np.ndarray) -> GraphScalers:
    node_scaler = StandardScaler().fit(X_all[train_idx].reshape(-1, X_all.shape[-1]))
    # Edge features: preserve availability and outage flag; scale only continuous features from index 2 onward.
    edge_scaler = StandardScaler().fit(E_all[train_idx, :, 2:].reshape(-1, E_all.shape[-1] - 2))
    global_scaler = StandardScaler().fit(G_all[train_idx])
    return GraphScalers(node_scaler, edge_scaler, global_scaler)


def transform_graph_arrays(indices: np.ndarray, scalers: GraphScalers) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    x = X_all[indices].copy()
    e = E_all[indices].copy()
    g = G_all[indices].copy()
    x = scalers.node_scaler.transform(x.reshape(-1, x.shape[-1])).reshape(x.shape)
    e_cont = scalers.edge_scaler.transform(e[:, :, 2:].reshape(-1, e.shape[-1] - 2)).reshape(e.shape[0], e.shape[1], e.shape[2] - 2)
    e[:, :, 2:] = e_cont
    g = scalers.global_scaler.transform(g)
    y = y_all[indices].reshape(-1, 1)
    s = sev_all[indices].reshape(-1, 1)
    return (
        torch.tensor(x, dtype=torch.float32, device=DEVICE),
        torch.tensor(e, dtype=torch.float32, device=DEVICE),
        torch.tensor(g, dtype=torch.float32, device=DEVICE),
        torch.tensor(y, dtype=torch.float32, device=DEVICE),
        torch.tensor(s, dtype=torch.float32, device=DEVICE),
    )

scalers = fit_graph_scalers(idx_train)
Xtr, Etr, Gtr, ytr, Str = transform_graph_arrays(idx_train, scalers)
Xva, Eva, Gva, yva, Sva = transform_graph_arrays(idx_val, scalers)
Xte, Ete, Gte, yte, Ste = transform_graph_arrays(idx_test, scalers)

add_check("preprocessed_train_finite", torch.isfinite(Xtr).all().item() and torch.isfinite(Etr).all().item() and torch.isfinite(Gtr).all().item())
add_check("edge_availability_kept_binary", set(torch.unique(Etr[:, :, 0]).cpu().numpy().round(6).tolist()).issubset({0.0, 1.0}))
add_check("edge_outage_flag_kept_binary", set(torch.unique(Etr[:, :, 1]).cpu().numpy().round(6).tolist()).issubset({0.0, 1.0}))

split_summary = pd.DataFrame([
    {"split": "train", "n": len(idx_train), "unsafe": int(y_all[idx_train].sum())},
    {"split": "validation", "n": len(idx_val), "unsafe": int(y_all[idx_val].sum())},
    {"split": "test", "n": len(idx_test), "unsafe": int(y_all[idx_test].sum())},
])
split_summary.to_csv(OUTDIR / "week7_split_summary.csv", index=False)
display(split_summary)

## 8. 原生 PyTorch message-passing GNN

模型结构：

```text
node features -> node encoder
edge features -> edge gate
message passing layers
mean pooling over nodes
global features concatenation
classification head + severity head
```

这里我们把 outaged line 的 message 直接乘以 `availability`：

\[
m_{ij} \leftarrow a_{ij}m_{ij}
\]

如果 line outage，则 \(a_{ij}=0\)。

In [ ]:
# ============================================================
# 8. Models
# ============================================================
class PhaseAwareGNN(nn.Module):
    def __init__(self, n_node_features: int, n_edge_features: int, n_global_features: int, hidden: int = 48, layers: int = 3):
        super().__init__()
        self.layers = layers
        self.node_encoder = nn.Sequential(nn.Linear(n_node_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.edge_gate = nn.Sequential(nn.Linear(n_edge_features, hidden), nn.Sigmoid())
        self.self_layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])
        self.msg_layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.readout = nn.Sequential(
            nn.Linear(hidden + n_global_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
        )
        self.cls_head = nn.Linear(hidden // 2, 1)
        self.sev_head = nn.Sequential(nn.Linear(hidden // 2, 1), nn.Softplus())

    def forward(self, x: torch.Tensor, edge_attr: torch.Tensor, global_feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: [B, N, F_node], edge_attr: [B, D, F_edge], global: [B, F_global]
        src = EDGE_INDEX[0].to(x.device)
        dst = EDGE_INDEX[1].to(x.device)
        h = self.node_encoder(x)
        availability = edge_attr[:, :, 0:1]
        for layer in range(self.layers):
            h_src = h[:, src, :]
            gate = self.edge_gate(edge_attr)
            messages = self.msg_layers[layer](h_src) * gate * availability
            agg = torch.zeros_like(h)
            agg.index_add_(1, dst, messages)
            h_next = self.self_layers[layer](h) + agg
            h = self.norms[layer](F.relu(h_next))
        pooled = h.mean(dim=1)
        z = self.readout(torch.cat([pooled, global_feat], dim=1))
        logits = self.cls_head(z)
        severity = self.sev_head(z)
        return logits, severity


class FlattenedMLP(nn.Module):
    def __init__(self, n_node_features: int, n_edge_features: int, n_global_features: int, hidden: int = 64):
        super().__init__()
        dim = len(BUS_NAMES) * n_node_features + (2 * len(LINE_SPECS)) * n_edge_features + n_global_features
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(0.05),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
        )
        self.cls_head = nn.Linear(hidden // 2, 1)
        self.sev_head = nn.Sequential(nn.Linear(hidden // 2, 1), nn.Softplus())

    def forward(self, x: torch.Tensor, edge_attr: torch.Tensor, global_feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        flat = torch.cat([x.flatten(1), edge_attr.flatten(1), global_feat], dim=1)
        z = self.net(flat)
        return self.cls_head(z), self.sev_head(z)


def composite_loss(logits: torch.Tensor, sev_pred: torch.Tensor, y: torch.Tensor, sev: torch.Tensor, pos_weight: float) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=torch.tensor([pos_weight], device=logits.device))
    # severity is nonnegative; scale mild to keep classification dominant.
    sev_loss = F.smooth_l1_loss(sev_pred, sev)
    # Probability-severity consistency: unsafe with positive severity should tend toward high probability.
    prob = torch.sigmoid(logits)
    sev_signal = torch.clamp(sev, 0.0, 1.0)
    consistency = F.mse_loss(prob, torch.maximum(y, sev_signal))
    return bce + 0.25 * sev_loss + 0.10 * consistency


def train_model(model: nn.Module, train_tensors, val_tensors=None, epochs: int = 60, lr: float = 2e-3, weight_decay: float = 1e-4, seed: int = SEED) -> Tuple[nn.Module, pd.DataFrame]:
    torch.manual_seed(seed)
    model = model.to(DEVICE)
    X_train, E_train, G_train, y_train, S_train = train_tensors
    pos = float(y_train.sum().item())
    neg = float(len(y_train) - pos)
    pos_weight = max(neg / max(pos, 1.0), 1.0)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits, sev_pred = model(X_train, E_train, G_train)
        loss = composite_loss(logits, sev_pred, y_train, S_train, pos_weight=pos_weight)
        loss.backward()
        opt.step()
        row = {"epoch": epoch + 1, "train_loss": float(loss.detach().cpu())}
        if val_tensors is not None:
            model.eval()
            with torch.no_grad():
                X_val, E_val, G_val, y_val, S_val = val_tensors
                v_logits, v_sev = model(X_val, E_val, G_val)
                v_loss = composite_loss(v_logits, v_sev, y_val, S_val, pos_weight=pos_weight)
                row["val_loss"] = float(v_loss.cpu())
        history.append(row)
    return model, pd.DataFrame(history)


def predict_model(model: nn.Module, tensors) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    X, E, G, y, S = tensors
    with torch.no_grad():
        logits, sev = model(X, E, G)
        prob = torch.sigmoid(logits).cpu().numpy().ravel()
        sev_pred = sev.cpu().numpy().ravel()
    return prob, sev_pred

# Forward and gradient proof on a small batch.
probe_model = PhaseAwareGNN(len(NODE_FEATURES), len(EDGE_FEATURES), len(GLOBAL_FEATURES), hidden=32, layers=2)
probe_logits, probe_sev = probe_model(Xtr[:4], Etr[:4], Gtr[:4])
add_check("gnn_forward_shape", tuple(probe_logits.shape) == (4, 1) and tuple(probe_sev.shape) == (4, 1))
probe_loss = composite_loss(probe_logits, probe_sev, ytr[:4], Str[:4], pos_weight=1.0)
probe_loss.backward()
all_grads_finite = all((p.grad is None) or torch.isfinite(p.grad).all().item() for p in probe_model.parameters())
add_check("gnn_finite_loss_and_gradients", torch.isfinite(probe_loss).item() and all_grads_finite, f"loss={float(probe_loss.detach())}")

print("Model proof checks passed.")

## 9. Permutation-invariance proof

Graph-level prediction should not depend on arbitrary bus ordering. If we permute node order and correctly remap edge indices, the GNN output should stay the same.

This proof is important because it distinguishes a graph model from a feature vector model that may accidentally learn bus index positions. In this implementation, node roles and features carry meaning; the ordering itself should not.

In [ ]:
# ============================================================
# 9. Permutation-invariance proof for graph-level model
# ============================================================
def permute_graph_tensors(x_one: torch.Tensor, e_one: torch.Tensor, perm: np.ndarray) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return permuted x and remapped EDGE_INDEX for one batch item.

    x_one: [1, N, F]
    e_one is unchanged because directed edges are still in the same edge list order, but the global EDGE_INDEX must be remapped.
    """
    inv = np.empty_like(perm)
    inv[perm] = np.arange(len(perm))
    x_perm = x_one[:, perm, :]
    edge_np = EDGE_INDEX.cpu().numpy().copy()
    edge_perm = torch.tensor(inv[edge_np], dtype=torch.long)
    return x_perm, e_one.clone(), edge_perm

class PhaseAwareGNNWithEdgeIndex(PhaseAwareGNN):
    """Same model but accepts an explicit edge index, used only for permutation proof."""
    def forward_with_edge_index(self, x, edge_attr, global_feat, edge_index):
        src = edge_index[0].to(x.device)
        dst = edge_index[1].to(x.device)
        h = self.node_encoder(x)
        availability = edge_attr[:, :, 0:1]
        for layer in range(self.layers):
            h_src = h[:, src, :]
            gate = self.edge_gate(edge_attr)
            messages = self.msg_layers[layer](h_src) * gate * availability
            agg = torch.zeros_like(h)
            agg.index_add_(1, dst, messages)
            h_next = self.self_layers[layer](h) + agg
            h = self.norms[layer](F.relu(h_next))
        pooled = h.mean(dim=1)
        z = self.readout(torch.cat([pooled, global_feat], dim=1))
        return self.cls_head(z), self.sev_head(z)

perm_model = PhaseAwareGNNWithEdgeIndex(len(NODE_FEATURES), len(EDGE_FEATURES), len(GLOBAL_FEATURES), hidden=32, layers=2).eval()
with torch.no_grad():
    logit0, sev0 = perm_model.forward_with_edge_index(Xtr[:1], Etr[:1], Gtr[:1], EDGE_INDEX)
    perm = np.random.default_rng(SEED).permutation(len(BUS_NAMES))
    x_perm, e_perm, edge_perm = permute_graph_tensors(Xtr[:1], Etr[:1], perm)
    logit1, sev1 = perm_model.forward_with_edge_index(x_perm, e_perm, Gtr[:1], edge_perm)
    diff = max(abs(float(logit0 - logit1)), abs(float(sev0 - sev1)))

add_check("permutation_invariance_untrained_gnn", diff < 1e-5, f"max_diff={diff:.3e}")
print("Permutation-invariance max diff:", diff)

## 10. Train flattened MLP and phase-aware GNN

我们训练两个模型：

1. **FlattenedMLP**：把全部 graph tensors 展平成一个 feature vector，作为非拓扑 baseline。
2. **PhaseAwareGNN**：显式使用 edge index、edge availability 和 message passing。

注意：这一步不是为了证明 GNN 一定比 MLP 好，而是为了建立可比较实验。小数据下性能波动很正常，重点是 protocol 和 proof。

In [ ]:
# ============================================================
# 10. Train holdout models
# ============================================================
train_tensors = (Xtr, Etr, Gtr, ytr, Str)
val_tensors = (Xva, Eva, Gva, yva, Sva)
test_tensors = (Xte, Ete, Gte, yte, Ste)

flat_model, flat_hist = train_model(
    FlattenedMLP(len(NODE_FEATURES), len(EDGE_FEATURES), len(GLOBAL_FEATURES), hidden=64),
    train_tensors, val_tensors, epochs=45, lr=2e-3, seed=SEED,
)
gnn_model, gnn_hist = train_model(
    PhaseAwareGNN(len(NODE_FEATURES), len(EDGE_FEATURES), len(GLOBAL_FEATURES), hidden=48, layers=3),
    train_tensors, val_tensors, epochs=55, lr=2e-3, seed=SEED + 1,
)

flat_hist["model"] = "FlattenedMLP"
gnn_hist["model"] = "PhaseAwareGNN"
training_curves = pd.concat([flat_hist, gnn_hist], ignore_index=True)
training_curves.to_csv(OUTDIR / "week7_training_curves.csv", index=False)

plt.figure(figsize=(7, 4))
for name, grp in training_curves.groupby("model"):
    plt.plot(grp["epoch"], grp["train_loss"], label=f"{name} train")
    if "val_loss" in grp.columns:
        plt.plot(grp["epoch"], grp["val_loss"], linestyle="--", label=f"{name} val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Week 7 training curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "week7_training_curves.png", dpi=180)
plt.show()

print("Finished training holdout models.")

## 11. High-recall threshold selection and holdout evaluation

阈值只能在 validation set 上选：

\[
\max_\tau \text{PFCallsSaved}_{val}(\tau)
\quad \text{s.t.}\quad
\text{Recall}_{val}(\tau)\ge 0.95
\]

然后把这个阈值固定，用在 test set 上报告。

In [ ]:
# ============================================================
# 11. Metrics and threshold selection
# ============================================================
def confusion_from_threshold(y_true: np.ndarray, prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_true = y_true.astype(int)
    y_pred = (prob >= threshold).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    fnr = fn / (tp + fn) if (tp + fn) else np.nan
    pf_saved = (tn + fn) / len(y_true) if len(y_true) else np.nan
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn, "recall": recall, "precision": precision, "FNR": fnr, "pf_calls_saved_fraction": pf_saved, "missed_violations": fn}


def select_high_recall_threshold(y_val: np.ndarray, prob_val: np.ndarray, target_recall: float = TARGET_RECALL) -> Tuple[float, pd.DataFrame]:
    candidates = np.unique(np.r_[0.0, prob_val, 1.0])
    rows = []
    for tau in candidates:
        m = confusion_from_threshold(y_val, prob_val, float(tau))
        rows.append({"threshold": float(tau), **m})
    tab = pd.DataFrame(rows).sort_values(["recall", "pf_calls_saved_fraction"], ascending=[False, False])
    feasible = tab[tab["recall"] >= target_recall]
    if feasible.empty:
        # Conservative fallback: threshold 0 sends all samples to exact PF, recall = 1.
        return 0.0, tab
    best = feasible.sort_values(["pf_calls_saved_fraction", "precision"], ascending=[False, False]).iloc[0]
    return float(best["threshold"]), tab


def eval_model_holdout(name: str, model: nn.Module) -> Tuple[Dict[str, float], pd.DataFrame, np.ndarray, np.ndarray]:
    p_val, s_val = predict_model(model, val_tensors)
    p_test, s_test = predict_model(model, test_tensors)
    tau, trade = select_high_recall_threshold(y_all[idx_val], p_val, TARGET_RECALL)
    metrics = confusion_from_threshold(y_all[idx_test], p_test, tau)
    try:
        auprc = average_precision_score(y_all[idx_test], p_test)
    except Exception:
        auprc = np.nan
    try:
        auroc = roc_auc_score(y_all[idx_test], p_test)
    except Exception:
        auroc = np.nan
    result = {"model": name, "selected_threshold": tau, "test_AUPRC": auprc, "test_AUROC": auroc, **metrics}
    trade["model"] = name
    return result, trade, p_test, s_test

flat_result, flat_trade, flat_p_test, flat_s_test = eval_model_holdout("FlattenedMLP", flat_model)
gnn_result, gnn_trade, gnn_p_test, gnn_s_test = eval_model_holdout("PhaseAwareGNN", gnn_model)

holdout_results = pd.DataFrame([flat_result, gnn_result])
screening_tradeoff = pd.concat([flat_trade, gnn_trade], ignore_index=True)
holdout_results.to_csv(OUTDIR / "week7_holdout_results.csv", index=False)
screening_tradeoff.to_csv(OUTDIR / "week7_screening_tradeoff.csv", index=False)

display(holdout_results)

# Manual confusion proof for selected GNN.
gnn_manual = confusion_from_threshold(y_all[idx_test], gnn_p_test, float(gnn_result["selected_threshold"]))
add_check("manual_confusion_matrix_keys", all(k in gnn_manual for k in ["TP", "FP", "TN", "FN", "recall", "FNR"]))
add_check("threshold_selected_from_validation_only", "selected_threshold" in gnn_result and np.isfinite(gnn_result["selected_threshold"]))

# Plots.
plt.figure(figsize=(6, 4))
for name, prob in [("FlattenedMLP", flat_p_test), ("PhaseAwareGNN", gnn_p_test)]:
    prec, rec, _ = precision_recall_curve(y_all[idx_test], prob)
    plt.plot(rec, prec, label=name)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Week 7 test precision-recall curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "week7_precision_recall_curves.png", dpi=180)
plt.show()

plt.figure(figsize=(6, 4))
for name, grp in screening_tradeoff.groupby("model"):
    plt.plot(grp["missed_violations"], grp["pf_calls_saved_fraction"], marker="o", linewidth=1, label=name)
plt.xlabel("Missed violations on validation set")
plt.ylabel("PF calls saved fraction")
plt.title("Week 7 screening tradeoff")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "week7_saved_calls_vs_missed_violations.png", dpi=180)
plt.show()

## 12. Scenario-group and contingency-group cross-validation

Random holdout is useful, but microgrid deployment needs OOD-style checks。

Scenario-group CV:

\[
\mathcal{S}_{train}\cap\mathcal{S}_{test}=\emptyset
\]

Contingency-group CV:

\[
\mathcal{C}_{train}\cap\mathcal{C}_{test}=\emptyset
\]

为了控制运行时间，CV 中训练较少 epochs。

In [ ]:
# ============================================================
# 12. Group CV utilities
# ============================================================
def train_eval_group_fold(test_mask: np.ndarray, group_name: str, fold_name: str, epochs: int = 90, seed: int = SEED) -> Tuple[Dict[str, float], np.ndarray]:
    test_idx = np.where(test_mask)[0]
    train_idx = np.where(~test_mask)[0]
    assert len(set(train_idx) & set(test_idx)) == 0
    # If train set only has one class, use conservative constant fallback. Should not happen for this dataset.
    if len(np.unique(y_all[train_idx])) < 2:
        prob = np.ones(len(test_idx))
        metrics = confusion_from_threshold(y_all[test_idx], prob, 0.5)
        return {"cv_type": group_name, "fold": fold_name, "n_test": len(test_idx), "threshold": 0.5, "AUPRC": np.nan, "AUROC": np.nan, **metrics}, prob
    sc = fit_graph_scalers(train_idx)
    Xtr_f, Etr_f, Gtr_f, ytr_f, Str_f = transform_graph_arrays(train_idx, sc)
    Xte_f, Ete_f, Gte_f, yte_f, Ste_f = transform_graph_arrays(test_idx, sc)
    model = PhaseAwareGNN(len(NODE_FEATURES), len(EDGE_FEATURES), len(GLOBAL_FEATURES), hidden=24, layers=1)
    model, _ = train_model(model, (Xtr_f, Etr_f, Gtr_f, ytr_f, Str_f), None, epochs=epochs, lr=2e-3, seed=seed)
    # Conservative threshold selected on training predictions for this compact teaching CV.
    p_train, _ = predict_model(model, (Xtr_f, Etr_f, Gtr_f, ytr_f, Str_f))
    tau, _ = select_high_recall_threshold(y_all[train_idx], p_train, TARGET_RECALL)
    prob, _ = predict_model(model, (Xte_f, Ete_f, Gte_f, yte_f, Ste_f))
    metrics = confusion_from_threshold(y_all[test_idx], prob, tau)
    try:
        auprc = average_precision_score(y_all[test_idx], prob) if len(np.unique(y_all[test_idx])) > 1 else np.nan
    except Exception:
        auprc = np.nan
    try:
        auroc = roc_auc_score(y_all[test_idx], prob) if len(np.unique(y_all[test_idx])) > 1 else np.nan
    except Exception:
        auroc = np.nan
    return {"cv_type": group_name, "fold": fold_name, "n_test": len(test_idx), "threshold": tau, "AUPRC": auprc, "AUROC": auroc, **metrics}, prob

# Scenario-group CV.
cv_rows = []
oof_prob_scenario = np.full(len(df), np.nan)
for k, scen in enumerate(sorted(df["scenario_id"].unique())):
    mask = df["scenario_id"].eq(scen).to_numpy()
    # Disjoint proof.
    train_scenarios = set(df.loc[~mask, "scenario_id"])
    test_scenarios = set(df.loc[mask, "scenario_id"])
    add_check(f"scenario_cv_disjoint_{scen}", len(train_scenarios & test_scenarios) == 0)
    row, prob = train_eval_group_fold(mask, "scenario_group", scen, epochs=18, seed=SEED + 10 + k)
    cv_rows.append(row)
    oof_prob_scenario[np.where(mask)[0]] = prob

# Contingency-group CV.
oof_prob_cont = np.full(len(df), np.nan)
for k, cont in enumerate(sorted(df["contingency_id"].unique())):
    mask = df["contingency_id"].eq(cont).to_numpy()
    train_cont = set(df.loc[~mask, "contingency_id"])
    test_cont = set(df.loc[mask, "contingency_id"])
    add_check(f"contingency_cv_disjoint_{cont}", len(train_cont & test_cont) == 0)
    row, prob = train_eval_group_fold(mask, "contingency_group", cont, epochs=18, seed=SEED + 100 + k)
    cv_rows.append(row)
    oof_prob_cont[np.where(mask)[0]] = prob

cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby("cv_type").agg(
    n_folds=("fold", "count"),
    mean_recall=("recall", "mean"),
    mean_precision=("precision", "mean"),
    mean_FNR=("FNR", "mean"),
    mean_pf_calls_saved=("pf_calls_saved_fraction", "mean"),
    total_missed_violations=("missed_violations", "sum"),
).reset_index()

add_check("scenario_oof_scores_finite", np.isfinite(oof_prob_scenario).all())
add_check("contingency_oof_scores_finite", np.isfinite(oof_prob_cont).all())

cv_results.to_csv(OUTDIR / "week7_group_cv_results.csv", index=False)
cv_summary.to_csv(OUTDIR / "week7_group_cv_summary.csv", index=False)

display(cv_summary)

plt.figure(figsize=(6, 4))
plt.bar(cv_summary["cv_type"], cv_summary["mean_recall"])
plt.ylabel("Mean recall")
plt.title("Week 7 OOD-style group CV recall")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUTDIR / "week7_group_cv_recall.png", dpi=180)
plt.show()

## 13. Top-k screening using scenario out-of-fold scores

Top-k evaluation should not use in-sample scores. Here we use scenario-group out-of-fold probabilities：每个 scenario 的 score 都来自没有见过该 scenario 的模型。

For each scenario:

1. rank contingencies by predicted risk;
2. take top \(k\);
3. compute violation recall@k。

In [ ]:
# ============================================================
# 13. Top-k ranking
# ============================================================
def topk_recall_table(scores: np.ndarray, max_k: int | None = None) -> pd.DataFrame:
    rows = []
    n_cont = df["contingency_id"].nunique()
    if max_k is None:
        max_k = n_cont
    for k in range(1, max_k + 1):
        recalls = []
        severe_hits = []
        for scen, grp in df.assign(score=scores).groupby("scenario_id"):
            grp = grp.sort_values("score", ascending=False)
            y = grp["violation_label"].to_numpy(dtype=int)
            s = grp["severity_score"].to_numpy(dtype=float)
            total_unsafe = y.sum()
            if total_unsafe == 0:
                continue
            top = grp.head(k)
            rec = top["violation_label"].sum() / total_unsafe
            recalls.append(rec)
            severe_idx = int(np.argmax(s))
            severe_cont = grp.iloc[severe_idx]["contingency_id"]
            severe_hits.append(1.0 if severe_cont in set(top["contingency_id"]) else 0.0)
        rows.append({
            "k": k,
            "pf_call_fraction": k / n_cont,
            "mean_violation_recall_at_k": float(np.mean(recalls)),
            "min_violation_recall_at_k": float(np.min(recalls)),
            "severe_hit_rate_at_k": float(np.mean(severe_hits)),
        })
    return pd.DataFrame(rows)

topk = topk_recall_table(oof_prob_scenario)
topk.to_csv(OUTDIR / "week7_topk_violation_recall.csv", index=False)
display(topk.head(12))

plt.figure(figsize=(6, 4))
plt.plot(topk["k"], topk["mean_violation_recall_at_k"], marker="o", label="mean recall@k")
plt.plot(topk["k"], topk["min_violation_recall_at_k"], marker="s", label="min recall@k")
plt.plot(topk["k"], topk["severe_hit_rate_at_k"], marker="^", label="severe hit@k")
plt.xlabel("k contingencies sent to exact 3ph PF")
plt.ylabel("Recall / hit rate")
plt.title("Week 7 top-k screening with scenario-OOD scores")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "week7_topk_violation_recall.png", dpi=180)
plt.show()

# Save predictions for later paper/report.
predictions = df[["row_id", "scenario_id", "contingency_id", "contingency_type", "violation_label", "severity_score"]].copy()
predictions["scenario_oof_gnn_prob"] = oof_prob_scenario
predictions["contingency_oof_gnn_prob"] = oof_prob_cont
predictions.to_csv(OUTDIR / "week7_oof_predictions.csv", index=False)

add_check("topk_table_finite", np.isfinite(topk.select_dtypes(include=[np.number]).to_numpy()).all())

## 14. Save validation summary and result summary

最后把所有 proof / cross-validation 结果保存为 CSV 和 JSON，方便 Week 8 写论文时直接引用。

In [ ]:
# ============================================================
# 14. Final validation summary and outputs
# ============================================================
validation_summary = pd.DataFrame(validation_records)
validation_summary.to_csv(OUTDIR / "week7_validation_summary.csv", index=False)

result_summary = {
    "n_samples": int(len(df)),
    "n_scenarios": int(df["scenario_id"].nunique()),
    "n_contingencies": int(df["contingency_id"].nunique()),
    "n_unsafe": int(df["violation_label"].sum()),
    "n_safe": int((1 - df["violation_label"]).sum()),
    "node_feature_dim": int(len(NODE_FEATURES)),
    "edge_feature_dim": int(len(EDGE_FEATURES)),
    "global_feature_dim": int(len(GLOBAL_FEATURES)),
    "best_holdout_model_by_saved_calls": str(holdout_results.sort_values(["missed_violations", "pf_calls_saved_fraction"], ascending=[True, False]).iloc[0]["model"]),
    "validation_checks_passed": int(validation_summary["passed"].sum()),
    "validation_checks_total": int(len(validation_summary)),
}
with open(OUTDIR / "week7_result_summary.json", "w", encoding="utf-8") as f:
    json.dump(result_summary, f, indent=2, ensure_ascii=False)

# Feature metadata.
pd.DataFrame({"node_feature": NODE_FEATURES}).to_csv(OUTDIR / "week7_node_feature_list.csv", index=False)
pd.DataFrame({"edge_feature": EDGE_FEATURES}).to_csv(OUTDIR / "week7_edge_feature_list.csv", index=False)
pd.DataFrame({"global_feature": GLOBAL_FEATURES}).to_csv(OUTDIR / "week7_global_feature_list.csv", index=False)

print(json.dumps(result_summary, indent=2, ensure_ascii=False))
display(validation_summary.tail(20))
assert validation_summary["passed"].all(), "Some validation checks failed."

## 如何解读 Week 7 结果

Week 7 的结果可能出现三种情况：

1. **GNN 优于 MLP**：说明拓扑归纳偏置在当前数据规模和 split 下有帮助。
2. **GNN 与 MLP 接近**：说明当前 tabular features 已经包含了足够多的拓扑/位置特征。
3. **GNN 劣于 MLP**：这在小数据下很常见，不代表图学习方向错误；它提示我们需要更多 operating scenarios、更丰富的 feeder 拓扑、phase-level graph 或更强的正则化。

论文写作时应避免夸大结论。更稳妥的表述是：

> We implemented a topology-aware phase-channel GNN screening pipeline with strict no-leakage validation and OOD-style group evaluation. On the small teaching dataset, the GNN serves as a structurally meaningful prototype; larger and more diverse scenario libraries are required to make claims about superiority over strong tabular baselines.


## 15. Student exercises

1. 把 GNN 的 `mean pooling` 改成 `max pooling` 或 `mean+max pooling`，比较 high-recall screening performance。
2. 去掉 `cont_line_endpoint` node flag，只保留 edge outage mask，观察 line outage 预测变化。
3. 去掉 A/B/C phase channels，只使用 total load / total PV，观察 VUF/stress scenario 表现。
4. 把 `PhaseAwareGNN` 的 hidden dimension 从 48 改为 16、32、96，比较 overfitting。
5. 用 scenario-group CV 的 out-of-fold scores 画每个 scenario 的 top-k ranking 表。
6. 思考：如果进入 phase-level GNN，每个 bus-phase node 的 edge 应如何定义？

## Week 7 到 Week 8 的衔接

Week 8 将把 Weeks 4–7 的结果组织成 mini-paper：

```text
Problem formulation: three-phase N-1 security screening
Dataset: scenario × contingency labels
Method: physics-informed MLP + phase-aware GNN
Evaluation: high recall, top-k ranking, OOD splits
```

本周的 GNN 结果可以作为论文方法部分的核心模型，也可以作为与 Week 6 PI-MLP 的对比。